In [1]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 443.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 11.0 MB/s eta 0:00:00


In [2]:
%pip -q install pillow-heif

from pathlib import Path
from PIL import Image, ImageOps
from pillow_heif import register_heif_opener

register_heif_opener()

root = Path("/content")
heic_files = [
    path for path in root.rglob("*")
    if path.is_file() and path.suffix.lower() in {".heic", ".heif"}
]

converted = 0
for source in heic_files:
    output = source.with_suffix(".jpg")
    if output.exists():
        print(f"Skipped (JPG exists): {output}")
        continue

    try:
        with Image.open(source) as image:
            image = ImageOps.exif_transpose(image).convert("RGB")
            image.save(output, "JPEG", quality=95)
        print(f"Converted: {source} → {output}")
        converted += 1
    except Exception as error:
        print(f"Failed: {source} — {error}")

print(f"Done: {converted} of {len(heic_files)} files converted.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 26.1 MB/s eta 0:00:00
Converted: /content/IMG_3005.HEIC → /content/IMG_3005.jpg
Converted: /content/IMG_3012.HEIC → /content/IMG_3012.jpg
Converted: /content/IMG_3013.HEIC → /content/IMG_3013.jpg
Converted: /content/IMG_3010.HEIC → /content/IMG_3010.jpg
Converted: /content/IMG_3004.HEIC → /content/IMG_3004.jpg
Converted: /content/IMG_3006.HEIC → /content/IMG_3006.jpg
Converted: /content/IMG_3003.HEIC → /content/IMG_3003.jpg
Converted: /content/IMG_3011.HEIC → /content/IMG_3011.jpg
Converted: /content/IMG_3007.HEIC → /content/IMG_3007.jpg
Converted: /content/IMG_3009.HEIC → /content/IMG_3009.jpg
Done: 10 of 10 files converted.


In [ ]:
from ultralytics import YOLO
import cv2

class ParkingDetector:
    # change min confidence here to test, default = 0.25
    def __init__(self, model_path="parking_detector.pt", vehicle_classes=None, min_confidence=0.25):
        self.model = YOLO(model_path)
        # The default COCO model with yolo has these types: Car=2, Motorcycle=3, Bus=5, Truck=7
        # If we want to sort by types of vehicles i added the option
        self.vehicle_classes = vehicle_classes if vehicle_classes else [2, 3, 5, 7]
        self.min_confidence = min_confidence

    def detect(self, image_path):
        results = self.model(image_path, conf=self.min_confidence, classes=self.vehicle_classes, verbose=False)
        return results

    def count_vehicles(self, results):
        # Pass list of vehicle classes
        return len(results[0].boxes)

    def get_detections(self, results):
        detections = []
        x1,y1,x2,y2 = 0,0,0,0
        delete = 0

        for box in results[0].boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0][:4])
            # print(f"x1: {x1}, x2: {x2}, y1: {y1}, y2: {y2}")
            area = (x2 - x1) * (y2 - y1)
            print(f"area: {area}")
            # change area for bounding box if you want different requirements
            if area <= 90000:
                delete += 1
                continue
            detections.append({
                "bbox": [x1, y1, x2, y2],
                "center": ((x1 + x2) // 2, (y1 + y2) // 2),
                "confidence": float(box.conf[0]),
                "class_id": int(box.cls[0])
            })

        return detections, delete

    def visualize(self, results, output_path="output.jpg"):
        result = results[0]
        keep = []

        for index, box in enumerate(result.boxes):
            x1, y1, x2, y2 = map(int, box.xyxy[0][:4])
            area = (x2 - x1) * (y2 - y1)
            # must also change here if you change the area above
            if area <= 90000:
                continue
            keep.append(index)

        result.update(boxes=result.boxes.data[keep])
        annotated = result.plot()
        cv2.imwrite(output_path, annotated)
        print(f"Saved annotated image to {output_path}")

In [ ]:
from pathlib import Path
from PIL import Image
from IPython.display import display

root = Path("/content")
output_dir = root / "vehicle_results"
output_dir.mkdir(exist_ok=True)

images = sorted(
    p for p in root.rglob("*")
    if p.is_file()
    and p.suffix.lower() in {".jpg"}
    and output_dir not in p.parents
)

if not images:
    raise FileNotFoundError("No JPG or PNG images found under /content.")

detector = ParkingDetector(model_path="yolov8n.pt")

for image_path in images:

    results = detector.detect(str(image_path))
    count = detector.count_vehicles(results)
    detections, delete = detector.get_detections(results)
    print(f"{image_path}: {count-delete} vehicles")

    print(f"Detections: {detections}")

    detector.visualize(results, output_path=str(output_dir / image_path.name))

area: 1539937
area: 2110872
area: 1318011
area: 1957824
area: 67063
area: 76704
/content/IMG_3003.jpg: 4 vehicles
Detections: [{'bbox': [2576, 2138, 3829, 3367], 'center': (3202, 2752), 'confidence': 0.8896045684814453, 'class_id': 2}, {'bbox': [743, 2398, 2621, 3522], 'center': (1682, 2960), 'confidence': 0.8757821321487427, 'class_id': 2}, {'bbox': [0, 2584, 1217, 3667], 'center': (608, 3125), 'confidence': 0.8405956029891968, 'class_id': 2}, {'bbox': [4058, 1800, 5706, 2988], 'center': (4882, 2394), 'confidence': 0.8015692830085754, 'class_id': 2}]
Saved annotated image to /content/vehicle_results/IMG_3003.jpg
area: 4450272
area: 6348608
area: 2547846
/content/IMG_3004.jpg: 3 vehicles
Detections: [{'bbox': [3252, 2469, 5708, 4281], 'center': (4480, 3375), 'confidence': 0.9446945786476135, 'class_id': 2}, {'bbox': [404, 1910, 3085, 4278], 'center': (1744, 3094), 'confidence': 0.8550678491592407, 'class_id': 2}, {'bbox': [0, 1613, 1314, 3552], 'center': (657, 2582), 'confidence': 0.79